In [17]:
# LIBS 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd
# %pip install emi-receiver
from emi_receiver import receiver
import ast
from handcalcs import  render
from handcalcs import  *
import handcalcs
#from localcode3 import *


# Initialization 
bl= '#1520c0' # blue 
rd= '#C62828' # red
fig_counter=1

## Transformer design
### Lm 

####  LLC ZVS condition

$$I_{mag,peak} \cdot t_{dead} \geq 2 C_{oss} \cdot V_{in}$$

With $I_{mag,peak} = \frac{V_{in}}{8 \cdot f \cdot L_m}$:

$$L_{m,max} = \frac{t_{dead}}{16 \cdot f \cdot C_{oss}}$$

####### LLC ZVS condition

$$I_{mag,peak} \cdot t_{dead} \geq 2 C_{oss} \cdot V_{in}$$

With $I_{mag,peak} = \frac{V_{in}}{8 \cdot f \cdot L_m}$:

$$L_{m,max} = \frac{t_{dead}}{16 \cdot f \cdot C_{oss}}$$


####### MOSFET DATASHEET

https://www.infineon.com/assets/row/public/documents/24/49/infineon-ipqc60t022s7-datasheet-en.pdf?fileId=8ac78c8c8eeb092c018fb9fed79a17e7

| Parameter                                      | Symbol  | Min. | Typ. | Max. | Unit | Note / Test Condition                                  |
|-----------------------------------------------|---------|------|------|------|------|--------------------------------------------------------|
| Input capacitance                              | Ciss    | —    | 5640 | —    | pF   | VGS = 0V, VDS = 300V, f = 250kHz                      |
| Output capacitance                             | Coss    | —    | 89   | —    | pF   | VGS = 0V, VDS = 300V, f = 250kHz                      |
| Effective output capacitance, energy related¹  | Co(er)  | —    | 302  | —    | pF   | VGS = 0V, VDS = 0 to 300V                             |
| Effective output capacitance, time related²    | Co(tr)  | —    | 2677 | —    | pF   | ID = constant, VGS = 0V, VDS = 0 to 300V              |
| Output charge                                  | Qoss    | —    | 803  | —    | nC   | VGS = 0V, VDS = 0 to 300V                             |

####### QUESTION

wich capa i must to take for ZVS condtions? make search in internet , it is very seroious i must have the correct answer


CHATGPT (the idiot one ): 302
CLAUDE + GEMINI : 2677

In [51]:
# verification 

# Qoss = 803 
V = 300
C = 1E3*Qoss/V
print("#" , C)

# my old design 
# IPW60R037P7
Cotr =Coss=  1599e-12# 1599pf 
tdead= 100e-9
Freq = 150
Lm_max = tdead / (16 * Freq*1e3 * Coss)
#print(1e6*Lm_max, "uH")

Lm_max = tdead / (16 * Freq*1e3 * Coss)
print(f"# Lm_max = {Lm_max*1e6:.1f} uH (for ZVS with Coss={Coss*1e12:.0f}pF, tdead={tdead*1e9:.0f}ns)")

# 2676.6666666666665
# Lm_max = 26.1 uH (for ZVS with Coss=1599pF, tdead=100ns)


In [32]:
#Coss = 100e-12  # F
Coss =  2677e-12  # F
# https://www.infineon.com/assets/row/public/documents/24/49/infineon-ipqc60t022s7-datasheet-en.pdf?fileId=8ac78c8c8eeb092c018fb9fed79a17e7
for Coss in (50e-12, 100e-12, 150e-12, 200e-12, 2677e-12 ): # F
    print(f"########### Coss = {Coss*1e12:.0f} pF")
    for tdead in (100e-9, 200e-9, 300e-9):       # s
        
        # LLC ZVS: Imag_peak * tdead >= 2*Coss*Vin
        # Imag_peak = Vin/(8*f*Lm)
        # => Lm_max = tdead / (16*f*Coss)
        Lm_max = tdead / (16 * Freq*1e3 * Coss)
        print(f"Lm_max = {Lm_max*1e6:.1f} uH (for ZVS with Coss={Coss*1e12:.0f}pF, tdead={tdead*1e9:.0f}ns)")

########### Coss = 50 pF
Lm_max = 1250.0 uH (for ZVS with Coss=50pF, tdead=100ns)
Lm_max = 2500.0 uH (for ZVS with Coss=50pF, tdead=200ns)
Lm_max = 3750.0 uH (for ZVS with Coss=50pF, tdead=300ns)
########### Coss = 100 pF
Lm_max = 625.0 uH (for ZVS with Coss=100pF, tdead=100ns)
Lm_max = 1250.0 uH (for ZVS with Coss=100pF, tdead=200ns)
Lm_max = 1875.0 uH (for ZVS with Coss=100pF, tdead=300ns)
########### Coss = 150 pF
Lm_max = 416.7 uH (for ZVS with Coss=150pF, tdead=100ns)
Lm_max = 833.3 uH (for ZVS with Coss=150pF, tdead=200ns)
Lm_max = 1250.0 uH (for ZVS with Coss=150pF, tdead=300ns)
########### Coss = 200 pF
Lm_max = 312.5 uH (for ZVS with Coss=200pF, tdead=100ns)
Lm_max = 625.0 uH (for ZVS with Coss=200pF, tdead=200ns)
Lm_max = 937.5 uH (for ZVS with Coss=200pF, tdead=300ns)
########### Coss = 2677 pF
Lm_max = 23.3 uH (for ZVS with Coss=2677pF, tdead=100ns)
Lm_max = 46.7 uH (for ZVS with Coss=2677pF, tdead=200ns)
Lm_max = 70.0 uH (for ZVS with Coss=2677pF, tdead=300ns)


#### ELP 102/20/38 with I 102/7/38


In [2]:
%%render 
Vin = 380 # V
Freq = 100 # kHz 
N= 4 # turns
A_mm2=534.2# mm2 - see datasheet
A = A_mm2 * 1e-6 # m2

T = 1/(Freq*1e3)

# 
# L_uH = 80
# half bridge 
Li = (Vin/2)*T/2 # v=L.di/dt 

Flux = Li /N

deltaB = Flux /A
Bpeak = deltaB/2 # T
Bpeak_mT = Bpeak * 1e3 # mT
# print("# Bpeak ", 1e3*Bpeak,"mT")
# Bpeak  222.29502059153876 mT

<IPython.core.display.Latex object>

In [3]:
# log : mT, kW/m3  = 220,700   100,80
mT =  np.array([220,100])
kW_per_m3 = np.array([700,80])
mTlog = np.log10(mT)
kWlog = np.log10(kW_per_m3)
a = (kWlog[1]-kWlog[0])/(mTlog[1]-mTlog[0])
b = kWlog[0] - a*mTlog[0]
a, b

(np.float64(2.751009514791533), np.float64(-3.5989290425911227))

In [4]:

Pv_kW=700 # kW/m3
P_nominal = 1200 # w
Pv = Pv_kW * 1000 # W/m3
Ve_mm3 = 67745 # mm3 - see datasheet
Ve = Ve_mm3 * 1e-9
P_loss= Pv * Ve # W
#P_lossRelative = 100*P_loss/P_nominal # \%
P_loss

47.42150000000001

In [5]:
AL_N97 = 9600 # nH/turns2

In [6]:
# AL_gap = f(AL_ungapped, g)
# 1/AL_gap = 1/AL_ungapped + g/(mu0 * Ae)
mu0 = 4 * np.pi * 1e-7  # H/m
Ae = A  # m2

g_mm = 1.0  # mm - air gap length
g = g_mm * 1e-3  # m

AL_ungapped = AL_N97 * 1e-9  # H/turns2 (from nH)
AL_gap = 1 / (1/AL_ungapped + g/(mu0 * Ae))  # H/turns2
AL_gap_nH = AL_gap * 1e9  # nH/turns2

print(f"g = {g_mm} mm")
print(f"AL_ungapped = {AL_N97} nH/turns2")
print(f"AL_gap = {AL_gap_nH:.1f} nH/turns2")

g = 1.0 mm
AL_ungapped = 9600 nH/turns2
AL_gap = 627.4 nH/turns2


In [7]:
g_mm = 0.4  # mm - air gap length
g = g_mm * 1e-3  # m
for N in range(4,16):
    Flux = Li /N

    deltaB = Flux /A
    Bpeak = deltaB/2 # T
    Bpeak_mT = Bpeak * 1e3 # mT

    kW_per_m3_est = 10**(a*np.log10(Bpeak_mT) + b)
    Pv_kW  = kW_per_m3_est # kW/m3
    Pv = Pv_kW * 1000 # W/m3
    P_loss_est = Pv * Ve # W

    LuH = AL_N97 * N**2 / 1e3 # uH
    DI = 1e6*Li/LuH
    # print("# N= ", N, "Bpeak ", Bpeak_mT,"mT", #"kW/m3 ", Pv, 
    #      "P_loss ", P_loss_est, "W", "Lu ", LuH, "uH", "DI ", DI   )



    AL_ungapped = AL_N97 * 1e-9  # H/turns2 (from nH)
    AL_gap = 1 / (1/AL_ungapped + g/(mu0 * Ae))  # H/turns2
    AL_gap_nH = AL_gap * 1e9  # nH/turns2
    LuH_gap = AL_gap_nH  * N**2 / 1e3 # uH

    print(f"# N= {N} Bpeak {Bpeak_mT:.2f} mT P_loss {P_loss_est:.2f} W Lu {LuH:.2f} uH DI {DI:.2f}A Lu_gap {LuH_gap:.2f} uH") 

# N= 4 Bpeak 222.30 mT P_loss 48.79 W Lu 153.60 uH DI 6.18A Lu_gap 22.86 uH
# N= 5 Bpeak 177.84 mT P_loss 26.41 W Lu 240.00 uH DI 3.96A Lu_gap 35.71 uH
# N= 6 Bpeak 148.20 mT P_loss 15.99 W Lu 345.60 uH DI 2.75A Lu_gap 51.43 uH
# N= 7 Bpeak 127.03 mT P_loss 10.47 W Lu 470.40 uH DI 2.02A Lu_gap 70.00 uH
# N= 8 Bpeak 111.15 mT P_loss 7.25 W Lu 614.40 uH DI 1.55A Lu_gap 91.42 uH
# N= 9 Bpeak 98.80 mT P_loss 5.24 W Lu 777.60 uH DI 1.22A Lu_gap 115.71 uH
# N= 10 Bpeak 88.92 mT P_loss 3.92 W Lu 960.00 uH DI 0.99A Lu_gap 142.85 uH
# N= 11 Bpeak 80.83 mT P_loss 3.02 W Lu 1161.60 uH DI 0.82A Lu_gap 172.85 uH
# N= 12 Bpeak 74.10 mT P_loss 2.38 W Lu 1382.40 uH DI 0.69A Lu_gap 205.71 uH
# N= 13 Bpeak 68.40 mT P_loss 1.91 W Lu 1622.40 uH DI 0.59A Lu_gap 241.42 uH
# N= 14 Bpeak 63.51 mT P_loss 1.55 W Lu 1881.60 uH DI 0.50A Lu_gap 279.99 uH
# N= 15 Bpeak 59.28 mT P_loss 1.29 W Lu 2160.00 uH DI 0.44A Lu_gap 321.42 uH


In [8]:
# N= 10 Bpeak 88.92 mT P_loss 3.92 W Lu 960.00 uH DI 0.99A Lu_gap 142.85 uH
# good compromise for ZVS and low losses. 

For ZVS you need enough magnetizing current to charge/discharge MOSFET Coss during dead time. If Lm is too high → insufficient Imag → loss of ZVS.

In [9]:
# So why are 10kW transformers physically bigger than 10W?
# Only because of wires:

#### Other core

In [52]:
data_cors = {
    "ELP 102/20/38 with I 102/7/38": {
        "AL": 9600, # nH/turns2, 
        "Ae": 534.2, # mm2
        "gapped": False
        }, 
    "E 42/21/15": {
        "AL": 3950 , # nH/turns2, 
        "Ae": 178, # mm2
        "gapped": False
        }, 
    "E 80/38/20 B66375G0000X187": {
        "AL": 4500 , # nH/turns2, 
        "Ae": 390, # mm2
        "gapped": False
         },
    "E 55/28/21 B66335G1000X187": {
        "AL": 496 , # nH/turns2,   
        "Ae": 354 , # mm2
        "gapped": True
         },
    "E 55/28/21DG B66335Q0100K187": {
        "AL": 100 , # nH/turns2,   
        "Ae": 354 , # mm2
        "gapped":True
         },
}



In [53]:
g_mm = 0.4*2  # mm - air gap length
g = g_mm * 1e-3  # m

arr= []
for core in data_cors:
    print("-"*10, core, "-"*10)
    AL = data_cors[core]["AL"]
    Ae_mm2 = data_cors[core]["Ae"]
    A = Ae_mm2 * 1e-6 # m2
    gapped= data_cors[core]["gapped"]


    for N in range(4,16,2):
        Flux = Li /N

        deltaB = Flux /A
        Bpeak = deltaB/2 # T
        Bpeak_mT = Bpeak * 1e3 # mT

        kW_per_m3_est = 10**(a*np.log10(Bpeak_mT) + b)
        Pv_kW  = kW_per_m3_est # kW/m3
        Pv = Pv_kW * 1000 # W/m3
        P_loss_est = Pv * Ve # W

        LuH = AL * N**2 / 1e3 # uH
        DI = 1e6*Li/LuH
        # print("# N= ", N, "Bpeak ", Bpeak_mT,"mT", #"kW/m3 ", Pv, 
        #      "P_loss ", P_loss_est, "W", "Lu ", LuH, "uH", "DI ", DI   )


        if not gapped:
            AL_ungapped = AL * 1e-9  # H/turns2 (from nH)
            AL_gap = 1 / (1/AL_ungapped + g/(mu0 * Ae))  # H/turns2
            AL_gap_nH = AL_gap * 1e9  # nH/turns2
            LuH_gap = AL_gap_nH  * N**2 / 1e3 # uH
            DI_gap = 1e6*Li/LuH_gap
        else:
            LuH_gap= np.nan
            DI_gap = np.nan
        dic = { "core": core,
                "N": N,
                "Bpeak_mT": Bpeak_mT,
                "P_loss_W": P_loss_est,
                "LuH": LuH,
                "DI_A": DI,
                "LuH_gap_uH": LuH_gap,
                "DI_gap_A": DI_gap
            }
        arr.append(dic)

        print(f"# N= {N} Bpeak {Bpeak_mT:.2f} mT P_loss {P_loss_est:.2f} W Lu {LuH:.2f} uH DI {DI:.2f}A Lu_gap {LuH_gap:.2f} uH DI_gap {DI_gap:.2f}A") 

---------- ELP 102/20/38 with I 102/7/38 ----------
# N= 4 Bpeak 222.30 mT P_loss 48.79 W Lu 153.60 uH DI 6.18A Lu_gap 12.35 uH DI_gap 76.94A
# N= 6 Bpeak 148.20 mT P_loss 15.99 W Lu 345.60 uH DI 2.75A Lu_gap 27.78 uH DI_gap 34.20A
# N= 8 Bpeak 111.15 mT P_loss 7.25 W Lu 614.40 uH DI 1.55A Lu_gap 49.39 uH DI_gap 19.24A
# N= 10 Bpeak 88.92 mT P_loss 3.92 W Lu 960.00 uH DI 0.99A Lu_gap 77.17 uH DI_gap 12.31A
# N= 12 Bpeak 74.10 mT P_loss 2.38 W Lu 1382.40 uH DI 0.69A Lu_gap 111.12 uH DI_gap 8.55A
# N= 14 Bpeak 63.51 mT P_loss 1.55 W Lu 1881.60 uH DI 0.50A Lu_gap 151.25 uH DI_gap 6.28A
---------- E 42/21/15 ----------
# N= 4 Bpeak 667.13 mT P_loss 1003.20 W Lu 63.20 uH DI 15.03A Lu_gap 11.07 uH DI_gap 85.79A
# N= 6 Bpeak 444.76 mT P_loss 328.82 W Lu 142.20 uH DI 6.68A Lu_gap 24.92 uH DI_gap 38.13A
# N= 8 Bpeak 333.57 mT P_loss 149.02 W Lu 252.80 uH DI 3.76A Lu_gap 44.29 uH DI_gap 21.45A
# N= 10 Bpeak 266.85 mT P_loss 80.66 W Lu 395.00 uH DI 2.41A Lu_gap 69.21 uH DI_gap 13.73A
# N= 12 Bpea

In [54]:
pd.DataFrame(arr)

,core,N,Bpeak_mT,P_loss_W,LuH,DI_A,LuH_gap_uH,DI_gap_A
0,ELP 102/20/38 with I 102/7/38,4,222.295021,48.794878,153.600,6.184896,12.346706,76.943599
1,ELP 102/20/38 with I 102/7/38,6,148.196680,15.993573,345.600,2.748843,27.780089,34.197155
2,ELP 102/20/38 with I 102/7/38,8,111.147510,7.248328,614.400,1.546224,49.386825,19.235900
3,ELP 102/20/38 with I 102/7/38,10,88.918008,3.923173,960.000,0.989583,77.166913,12.310976
4,ELP 102/20/38 with I 102/7/38,12,74.098340,2.375796,1382.400,0.687211,111.120355,8.549289
5,ELP 102/20/38 with I 102/7/38,14,63.512863,1.554669,1881.600,0.504889,151.247150,6.281110
6,E 42/21/15,4,667.134831,1003.198176,63.200,15.031646,11.073507,85.790348
7,E 42/21/15,6,444.756554,328.819833,142.200,6.680731,24.915390,38.129044
8,E 42/21/15,8,333.567416,149.021988,252.800,3.757911,44.294027,21.447587
9,E 42/21/15,10,266.853933,80.658471,395.000,2.405063,69.209417,13.726456


In [ ]:
# E 80/38/20 B66375G0000X187
# N= 12 Bpeak 101.50 mT P_loss 5.65 W Lu 648.00 uH DI 1.47A Lu_gap 176.02 uH DI_gap 5.40A 


# ---------- E 55/28/21 B66335G1000X187 ----------
# N= 12 Bpeak 111.82 mT P_loss 7.37 W Lu 71.42 uH DI 13.30A

### Lr

In [14]:


data_cors = {
    "E 42/21/15": {
        "AL": 3950 , # nH/turns2, 
        "Ae": 178, # mm2
        "gapped": False
        },
    "E 32/16/11DG B66233Q0100K187": {
        "AL": 100 , # nH/turns2,   
        "Ae": 97 , # mm2
        "gapped": True
         },
        "ETD 39/20/13DG B66363Q0150K187":{
        "AL": 150 , # nH/turns2,   
        "Ae": 125 , # mm2
        "gapped": True
         },
}



g_mm = 0.4  # mm - air gap length
g = g_mm * 1e-3  # m


for core in data_cors:
    print("-"*10, core, "-"*10)
    AL = data_cors[core]["AL"]
    Ae_mm2 = data_cors[core]["Ae"]
    A = Ae_mm2 * 1e-6 # m2
    gapped= data_cors[core]["gapped"]


    for N in range(4,16,2):
        Flux = Li /N

        deltaB = Flux /A
        Bpeak = deltaB/2 # T
        Bpeak_mT = Bpeak * 1e3 # mT

        kW_per_m3_est = 10**(a*np.log10(Bpeak_mT) + b)
        Pv_kW  = kW_per_m3_est # kW/m3
        Pv = Pv_kW * 1000 # W/m3
        P_loss_est = Pv * Ve # W

        LuH = AL * N**2 / 1e3 # uH
        DI = 1e6*Li/LuH
        # print("# N= ", N, "Bpeak ", Bpeak_mT,"mT", #"kW/m3 ", Pv, 
        #      "P_loss ", P_loss_est, "W", "Lu ", LuH, "uH", "DI ", DI   )


        if not gapped:
            AL_ungapped = AL * 1e-9  # H/turns2 (from nH)
            AL_gap = 1 / (1/AL_ungapped + g/(mu0 * Ae))  # H/turns2
            AL_gap_nH = AL_gap * 1e9  # nH/turns2
            LuH_gap = AL_gap_nH  * N**2 / 1e3 # uH
            DI_gap = 1e6*Li/LuH_gap
        else:
            LuH_gap= np.nan
            DI_gap = np.nan

        print(f"# N= {N} Bpeak {Bpeak_mT:.2f} mT P_loss {P_loss_est:.2f} W Lu {LuH:.2f} uH DI {DI:.2f}A Lu_gap {LuH_gap:.2f} uH DI_gap {DI_gap:.2f}A") 

---------- E 42/21/15 ----------
# N= 4 Bpeak 667.13 mT P_loss 1003.20 W Lu 63.20 uH DI 15.03A Lu_gap 18.85 uH DI_gap 50.41A
# N= 6 Bpeak 444.76 mT P_loss 328.82 W Lu 142.20 uH DI 6.68A Lu_gap 42.40 uH DI_gap 22.40A
# N= 8 Bpeak 333.57 mT P_loss 149.02 W Lu 252.80 uH DI 3.76A Lu_gap 75.38 uH DI_gap 12.60A
# N= 10 Bpeak 266.85 mT P_loss 80.66 W Lu 395.00 uH DI 2.41A Lu_gap 117.78 uH DI_gap 8.07A
# N= 12 Bpeak 222.38 mT P_loss 48.85 W Lu 568.80 uH DI 1.67A Lu_gap 169.61 uH DI_gap 5.60A
# N= 14 Bpeak 190.61 mT P_loss 31.96 W Lu 774.20 uH DI 1.23A Lu_gap 230.85 uH DI_gap 4.12A
---------- E 32/16/11DG B66233Q0100K187 ----------
# N= 4 Bpeak 1224.23 mT P_loss 5329.49 W Lu 1.60 uH DI 593.75A Lu_gap nan uH DI_gap nanA
# N= 6 Bpeak 816.15 mT P_loss 1746.86 W Lu 3.60 uH DI 263.89A Lu_gap nan uH DI_gap nanA
# N= 8 Bpeak 612.11 mT P_loss 791.68 W Lu 6.40 uH DI 148.44A Lu_gap nan uH DI_gap nanA
# N= 10 Bpeak 489.69 mT P_loss 428.50 W Lu 10.00 uH DI 95.00A Lu_gap nan uH DI_gap nanA
# N= 12 Bpeak 408

### References

<a id="referencesID4848878444887dd1"></a> [1] [TDK, Ferrites and accessories, SIFERRIT material N97](https://www.tdk-electronics.tdk.com/download/528886/81166f0de556e5b6a94db7793daed936/pdf-n97.pdf)

<a id="referencesID4848878444887dd2"></a> [2] [TDK, ELP 102/20/38 with I 102/7/38 Datasheet](https://www.mouser.com/datasheet/2/400/elp_102_20_38-1527691.pdf)




